## Mount Your Google Drive

In [1]:
from google.colab import drive
drive.mount('/gdrive')

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).


# Setup

In [1]:
# !pip uninstall -y transformers peft
# !pip install transformers==4.57.1
# !pip install peft==0.17.1
# !pip install accelerate==1.8.1
# #----------------------------------------------------------------------
# !pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
# !pip install -qU openai==1.61.0 wandb
# !pip install groq


  Using cached accelerate-1.8.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.8.1-py3-none-any.whl (365 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llamafactory 0.9.6.dev0 requires peft<=0.18.1,>=0.18.0, but you have peft 0.17.1 which is incompatible.


In [2]:
!pip install -qU transformers==4.57.1 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install groq
!pip install peft==0.17.1
!pip install accelerate==1.8.1

In [3]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.6.dev0-py3-none-any.whl size=26915 sha256=d8012c7ebc2e9eaf2b9f3fc8c681c4574fd0a5f8df06de7e008ff97f22b0381a
  Stored in directory: /tmp/pip-ephem-wheel-cache-5622i7ri/wheels/68/8b/5e/52f9888e6a91a2651260d603137c052b925af896da6e32a3f7
Successfully built llamafactory
  Attempting uninstall: peft
    Found existing installation: peft 0.17.1
    Uninstalling peft-0.17.1:
      Successfully 

In [4]:
from google.colab import userdata
import wandb  # for model evaluation & traking
from huggingface_hub import login


wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('HF')
# !huggingface-cli login -- token {hf_token}
login(token=hf_token)
!huggingface-cli whoami

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sherif-911 (sherif-911-huawei) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
sherif-911


In [5]:
!pip install json_repair
import os
import json
from os.path import join
import random
from tqdm.auto import tqdm # for making progress par
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime
import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

data_dir = "/gdrive/MyDrive/Fine-tuning LLM/Data"
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda"#"cpu"
torch_dtype = None

# Tasks

In [6]:
story = """
ذكرت مجلة فوربس أن العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال،
 حيث تتأثر هذه العلاقة بأنماط السلوك المالي المتوارثة عبر الأجيال.

التقرير الذي يستند إلى أبحاث الأستاذ الجامعي شاين إنيت حول
الرفاه المالي يوضح أن لكل شخص "شخصية مالية" تتحدد وفقا لطريقة
 تفاعله مع المال، والتي تتأثر بشكل مباشر بتربية الأسرة وتجارب الطفولة.

 الأبعاد الثلاثة للعلاقة بالمال
بحسب الدراسة، هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال:

الاكتساب (A): يميل الأفراد الذين ينتمون لهذا
 البعد إلى اعتبار المال سلعة قابلة للجمع، حيث يرون
في تحقيق الثروة هدفا بحد ذاته. والجانب السلبي لهذا
 النمط هو إمكانية التحول إلى هوس بالثروة أو العكس،
 أي رفض تام لاكتساب المال باعتباره مصدرا للفساد.

الاستخدام (U): يرى هؤلاء الأشخاص المال أداة للتمتع بالحياة، حيث يربطون قيمته بقدرته على توفير
المتعة والراحة. ومع ذلك، قد يصبح
البعض مدمنا على الإنفاق، في حين يتجه آخرون إلى التقشف المفرط خوفا من المستقبل.

الإدارة (M): أصحاب هذا النمط يعتبرون المال مسؤولية تتطلب التخطيط الدقيق. لكن في بعض الحالات،
 قد يتحول الأمر إلى هوس مفرط بإدارة الإنفاق، مما يؤثر سلبا على العلاقات الشخصية.

 كيف تؤثر العائلة على علاقتنا بالمال؟
يشير التقرير إلى أن التجارب الأسرية تلعب دورا رئيسيا في تحديد
 "الشخصية المالية" لكل فرد، على سبيل المثال، إذا كان أحد الوالدين يعتمد على المال
كمكافأة للسلوك الجيد، فقد يتبنى الطفل لاحقا النمط نفسه في حياته البالغة.

لتحليل هذه التأثيرات بشكل دقيق، طورت رابطة العلاج المالي
(Financial Therapy Association) أداة تسمى مخطط الجينوم المالي (Money Genogram)،
وهو نموذج يُستخدم لتحديد الأنماط المالية داخل العائلة.

تتضمن هذه الأداة:

رسم شجرة عائلية.
تصنيف أفراد العائلة وفقا للأبعاد الثلاثة للعلاقة بالمال (A ،U ،M).
تحديد ما إذا كان السلوك المالي لكل فرد صحيا (+) أو غير صحي (-).
على سبيل المثال، إذا نشأ شخص في عائلة
اعتادت على الإنفاق المفرط، فقد يكون لديه ميل قوي إلى اتباع النمط نفسه،
 أو العكس تماما، حيث يصبح مقتصدا بشكل مبالغ فيه كرد فعل نفسي.
"""

### Details Extraction

In [7]:
# {
#     "story_title" : "",
#     "story_keywords" : ["kw_1","kw_2"],
#     "story_summary" : ["...",",,,,"],
#     "story_category" : "category",
#     "story_entities" [
#         {
#             "entity_value" : "القاهره",
#             "entity_type" : "location"
#         }
#     ]
# }

StoryCategoryType = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"
]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"
]

class Entity(BaseModel):
  entity_value: str = Field(...,
                      description="The actual name or value of the entity.")
  entity_type: EntityType = Field(..., description="The type of recognized entity.")

class NewsDetails(BaseModel):
  story_title: str = Field(...,min_length=5 , max_length=300
                                 ,description="A fully informative and SEO optimized title of the story.")

  story_keywords: List[str] = Field(..., min_items=1, max_items=5,
                                  description="Relevant keywords associated with the story.")

  story_summary: List[str] = Field(..., min_items=1, max_items=5,
                                   description="Summarized key points about the story (1-5 points).")

  story_category: StoryCategoryType = Field(...,
                                   description="Category of the news story.")

  story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                   description="List of identified entities in the story.")
#

/tmp/ipykernel_16700/2152963745.py:34: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=1, max_items=5,
/tmp/ipykernel_16700/2152963745.py:34: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=1, max_items=5,
/tmp/ipykernel_16700/2152963745.py:37: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(..., min_items=1, max_it

In [8]:
details_extraction_messages = [
    {
        "role" : "system",
        "content" : "\n".join([
            "You are an NLP data paraser.",
            "You will be provided by an Arabic text associated with a Pydantic scheme.",
            "Generate the ouptut in the same story language.",
            "You have to extract JSON details from text according the Pydantic details.",
            "Extract details as mentioned in text.",
            "Do not generate any introduction or conclusion."
        ])
    },
    {
        "role" : "system",
        "content" : "\n".join([
            "## Story:",
            story.strip(),
            "",
            "## Pydantic Scheme:",
            json.dumps(
                NewsDetails.model_json_schema(),ensure_ascii=False
                ),
            "",
            "## Story Details:",
            "```json"
        ])
    }
]

### Translation

In [9]:
# {
#     "translated_title" : "Translated title,
#     "translated_text" : "Translated text,
# }

targeted_lang = "English"

class TranslatedStory(BaseModel):
    translated_title: str = Field(...,min_length=5 , max_length=300
                                  ,description="Suggested translated title of the news story.")
    translated_text: str = Field(...,min_length=5
                                  ,description="Translated content of the news story.")

translated_messages = [
      {
          "role" : "system",
          "content" : "\n".join([
            "You are a professional translator.",
            "You will be provided by an Arabic text.",
            "You have to translate the text into the `Targeted Language`.",
            "Follow the provided Scheme to generate a JSON",
            "Do not generate any introduction or conclusion.",
            "return only json file."
          ])
      },
      {
          "role" : "user",
          "content" : "\n".join([
            "## Story:",
            story.strip(),
            "",
            "## Pydantic Scheme:",
            json.dumps(
                TranslatedStory.model_json_schema(),ensure_ascii=False
                ),
            "",
            "## Targeted Language:",
            targeted_lang,
            "",
            "## Story Details:",
            "```json"
          ])
      }
  ]


# Evaluation

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map= "auto",
    torch_dtype= torch_dtype,
    # load_in_8bit=True,
    # trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


KeyboardInterrupt: 

# Model

In [ ]:
text = tokenizer.apply_chat_template(
    details_extraction_messages,
    tokenize=False,
    add_generation_prompt=True,
    # return_tensors="pt"
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# model start gnerate text.
generated_ids = model.generate(
    model_inputs.input_ids,
    attention_mask=model_inputs.attention_mask, # Fix: Pass attention_mask
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)
# get the generated IDs only.
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

KeyboardInterrupt: 

In [ ]:
# text
# model_inputs
# generated_ids
print(response)

['{\n  "story_title": "How Family Influences Financial Behavior",\n  "story_keywords": [\n    "family influence",\n    "financial behavior",\n    "moneymaking",\n    "money management",\n    "money genogram"\n  ],\n  "story_summary": [\n    "Families play a crucial role in shaping individuals\' financial behaviors.",\n    "Individuals inherit certain financial traits from their families through generations."\n  ],\n  "story_category": "economy",\n  "story_entities": [\n    {\n      "entity_value": "Shawn Eini",\n      "entity_type": "person-male"\n    },\n    {\n      "entity_value": "Financial Therapy Association",\n      "entity_type": "organization"\n    }\n  ]\n}']


In [ ]:
text = tokenizer.apply_chat_template(
    translated_messages,
    tokenize=False,
    add_generation_prompt=True,
    # return_tensors="pt"
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# model start gnerate text.
generated_ids = model.generate(
    model_inputs.input_ids,
    attention_mask=model_inputs.attention_mask, # Fix: Pass attention_mask
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)
# get the generated IDs only.
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

In [ ]:
print(response)

['{\n  "translated_title": "Forbes Magazine Reveals Family Plays a Central Role in Forming Individuals\' Financial Relationships",\n  "translated_text": "Forbes magazine has revealed that family plays a central role in shaping individuals\' financial relationships, as these relationships are influenced by inherited behavioral patterns across generations."\n}']


# Evaluation OpenAI or groq [ Lamma 3.3 70B ]

In [ ]:
# from openai import OpenAI
# from google.colab import userdata

# openai_client = OpenAI(
#     api_key=userdata.get('openai-colab'),
#     # base_url="http://localhost:8000"
# )

# openai_model_id = "gpt-4o-mini"

In [ ]:
# chat_completion = openai_client.chat.completions.create(
#     messages=details_extraction_messages,
#     model=openai_model_id,
#     temperature=0.2,
# )

# print(chat_completion.choices[0].message.content)

In [ ]:
# chat_completion = openai_client.chat.completions.create(
#     messages=translation_messages,
#     model=openai_model_id,
#     temperature=0.2,
# )

# print(chat_completion.choices[0].message.content)

In [ ]:
# !pip install groq

# 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
# llama-3.1-8b-instant
# llama-3.3-70b-versatile
# llama3-70b-8192
# llama3-8b-8192

from groq import Groq
from google.colab import userdata

groq_client = Groq(
    api_key=userdata.get('groq'), # Assuming 'groq' is the key for your Groq API token
)

groq_model_id = "llama-3.3-70b-versatile" # Corrected model ID for Llama 3 70B on Groq


In [ ]:
chat_completion = groq_client.chat.completions.create(
    messages=details_extraction_messages,
    model=groq_model_id,
    temperature=0.2,
)

print(chat_completion.choices[0].message.content)

```json
{
  "story_title": "العلاقة بين الأسرة والمال",
  "story_keywords": [
    "العلاقة بالمال",
    "شخصية مالية",
    "أبعاد العلاقة بالمال",
    "تأثير العائلة على المال"
  ],
  "story_summary": [
    "العلاقة بين الأسرة والمال تعتمد على أنماط السلوك المالي المتوارثة عبر الأجيال",
    "ثلاثة أبعاد رئيسية تشكل علاقتنا بالمال: الاكتساب، الاستخدام، والإدارة",
    "التجارب الأسرية تلعب دورا رئيسيا في تحديد الشخصية المالية لكل فرد",
    "مخطط الجينوم المالي أداة لتحديد الأنماط المالية داخل العائلة",
    "التأثيرات الأسرية يمكن أن تؤثر على السلوك المالي الصحي أو غير الصحي"
  ],
  "story_category": "economy",
  "story_entities": [
    {
      "entity_value": "مجلة فوربس",
      "entity_type": "organization"
    },
    {
      "entity_value": "شاين إنيت",
      "entity_type": "person-male"
    },
    {
      "entity_value": "الرفاه المالي",
      "entity_type": "not_specified"
    },
    {
      "entity_value": "رابطة العلاج المالي",
      "entity_type": "organization"
    },
    {
     

In [ ]:
chat_completion = groq_client.chat.completions.create(
    messages=translated_messages,
    model=groq_model_id,
    temperature=0.2,
)

print(chat_completion.choices[0].message.content)

```json
{
  "translated_title": "The Role of Family in Shaping Our Relationship with Money",
  "translated_text": "According to Forbes magazine, family plays a central role in shaping individuals' relationship with money, as this relationship is influenced by inherited financial behavior patterns across generations. A report based on Professor Shane Enete's research on financial well-being explains that each person has a 'financial personality' determined by how they interact with money, which is directly influenced by family upbringing and childhood experiences. The study identifies three main dimensions that shape our relationship with money: Acquisition (A), where individuals tend to view money as a collectible commodity and see wealth accumulation as a goal in itself, but may become obsessed with wealth or reject it altogether; Utilization (U), where people see money as a tool for enjoying life and link its value to the pleasure and comfort it provides, but may become addicted to s

## **Knowledge Distillation**


In [ ]:
raw_data_path = join(data_dir , "news-sample.jsonl")

raw_data = []
for line in open(raw_data_path):
  if line.strip() == "":
    continue

  raw_data.append(
      json.loads(line.strip())
  )

random.Random(45).shuffle(raw_data)

print(f"Total Samples: {len(raw_data)}")

Total Samples: 2400


In [ ]:
raw_data[0]['content']

'أثار تنظيم امتحانات الشهادة السودانية الثانوية العامة هذا العام جدلا واسعا لأسباب عديدة تتصل بالوضع الأمني المتدهور والحرب المستمرة في السودان منذ أبريل 2023، والتي تغطي رقعة واسعة من مناطق البلاد. \n وترتب على ذلك عمليا إجراء الامتحانات في الأماكن الآمنة التي يسيطر عليها الجيش، وتعذرها في مناطق انتشار قوات الدعم السريع في دارفور والجزيرة والخرطوم، ما تسبب في حرمان أعداد من الطلبة من الجلوس للامتحانات تقدر بـ30 من إجمالي الذين سجلوا لجلوسها قبل اندلاع الحرب بحسب تقديرات وزارة التربية والتعليم. \n وقد أفضى هذا الوضع إلى تباين الموقف في الفضاء العام من إجراء الامتحانات في ظل هذه الحرب واختلاف في وجهات النظر، فالإجراء في نظر المعترضين عليه شكل من أشكال التمييز وعدم المساواة في فرص أداء الامتحانات، ويثير مخاوف تتعلق بجودة وعدالة الامتحانات واحتمال تسرب أسئلتها بسبب الظروف الأمنية غير المستقرة، وهو لا يراعي عدم توفر الظروف المناسبة للطلبة للتحضير الجيد لها، وأن وصولهم إلى مراكز الامتحانات يعرضهم لمخاطر أمنية، وأن الامتحانات قد تتحول -هي الأخرى- لأداة تكرس واقع الحرب. \n وكنتيجة لهذه الآراء

In [ ]:
import time

cloud_model_id = "llama-3.3-70b-versatile"
price_per_1m_input_tokens = 0.150
price_per_1m_output_tokens = 0.600

prompt_tokens = 0
completion_tokens = 0

ix = 0

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

save_to = join(data_dir, "stf.jsonl")

for story in tqdm(raw_data):
    sample_details_extraction_messages = [
        {
            "role": "system",
            "content": "\n".join([
                "You are an NLP data paraser.",
                "You will be provided by an Arabic text associated with a Pydantic scheme.",
                "Generate the ouptut in the same story language.",
                "You have to extract JSON details from text according the Pydantic details.",
                "Extract details as mentioned in text.",
                "Do not generate any introduction or conclusion."
            ])
        },
        {
            "role": "user",
            "content": "\n".join([
                "## Story:",
                story['content'].strip(),
                "",

                "## Pydantic Details:",
                json.dumps(
                    NewsDetails.model_json_schema(), ensure_ascii=False
                ),
                "",

                "## Story Details:",
                "```json"
            ])
        }
    ]

    response = groq_client.chat.completions.create(
        messages=sample_details_extraction_messages,
        model=cloud_model_id,
        temperature=0.2,
    )

    # Introduce a delay to avoid rate limiting
    time.sleep(1.5)

    if response.choices[0].finish_reason != "stop":
      # print(response.choices[0].finish_reason)
      prompt_tokens += response.usage.prompt_tokens
      continue

    llm_response = response.choices[0].message.content
    llm_resp_dict = parse_json(llm_response)

    if not llm_resp_dict:
      # print(llm_response)
      continue

    with open(save_to, "a", encoding="utf8") as f:
      f.write(json.dumps({
          "id":ix,
          "story": story['content'].strip(),
          "task": "Extrat the story details into a JSON.",
          "output_scheme": json.dumps( NewsDetails.model_json_schema(), ensure_ascii=False ),
          "response": llm_resp_dict,
        }, ensure_ascii=False, default=str)  + "\n" )

    ix += 1
    prompt_tokens += response.usage.prompt_tokens
    completion_tokens += response.usage.completion_tokens

    if(ix % 3) == 0:
        cost_input = (prompt_tokens / 1_000_000) * price_per_1m_input_tokens
        cost_output = (completion_tokens / 1_000_000) * price_per_1m_output_tokens
        total_cost = cost_input + cost_output

        print(f"Iteration {ix}: Total Cost = ${total_cost:.4f} ")

  0%|          | 0/2400 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k8xfe9vqfsxt6e8ern003hhc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96619, Requested 6936. Please try again in 51m11.52s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
cloud_model_id = "llama-3.3-70b-versatile"
price_per_1m_input_tokens = 0.150
price_per_1m_output_tokens = 0.600

prompt_tokens = 0
completion_tokens = 0

save_to = join(data_dir, "datasets", "sft.jsonl")

ix = 0
for story in tqdm(raw_data):

    for targeted_lang in ["English", "French"]:
        sample_translation_messages = [
            {
                "role": "system",
                "content": "\n".join([
                    "You are a professional translator.",
                    "You will be provided by an Arabic text.",
                    "You have to translate the text into the `Targeted Language`.",
                    "Follow the provided Scheme to generate a JSON",
                    "Do not generate any introduction or conclusion."
                ])
            },
            {
                "role": "user",
                "content": "\n".join([
                    "## Pydantic Details:",
                    json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
                    "",

                    "## Targeted Language or Dialect:",
                    targeted_lang,
                    "",

                    "## Story:",
                    story['content'].strip(),
                    "",

                    "## Translated Story:",
                    "```json"
                ])
            }
        ]

        response = groq_client.chat.completions.create(
                                messages=sample_translation_messages,
                                model=cloud_model_id,
                                temperature=0.2,
                            )

        if response.choices[0].finish_reason != "stop":
            prompt_tokens += response.usage.prompt_tokens
            continue

        llm_response = response.choices[0].message.content
        llm_resp_dict = parse_json(llm_response)

        if not llm_resp_dict:
            continue

        with open(save_to, "a", encoding="utf8") as dest:
            dest.write(json.dumps({
                "id": ix,
                "story": story['content'].strip(),

                "output_scheme": json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
                "task": f"You have to translate the story content into {targeted_lang} associated with a title into a JSON.",

                "response": llm_resp_dict,
            }, ensure_ascii=False, default=str)  + "\n" )

        ix += 1
        prompt_tokens += response.usage.prompt_tokens
        completion_tokens += response.usage.completion_tokens

        if(ix % 3) == 0:
            cost_input = (prompt_tokens / 1_000_000) * price_per_1m_input_tokens
            cost_output = (completion_tokens / 1_000_000) * price_per_1m_output_tokens
            total_cost = cost_input + cost_output

            print(f"Iteration {ix}: Total Cost = ${total_cost:.4f} ")

# Format Finetuning Datasets

In [ ]:
sft_data_path = join(data_dir, "sft.jsonl")
llm_finetuning_data = []

system_message = "\n".join([
    "You are a professional NLP data parser.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
    if line.strip() == "":
        continue

    rec = json.loads(line.strip())

    llm_finetuning_data.append({
        "system":system_message,
        "instruction":"\n".join([
            "## Story:",
            rec["story"],
            "",
            "## Task:",
            rec["task"],
            "",
            "## Output Scheme:",
            rec["output_scheme"],
            "",
            "## Output JSON:",
            "```json"
        ]),
        "input":"",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

random.Random(45).shuffle(llm_finetuning_data)

In [ ]:
# rec
# llm_finetuning_data[0]
len(llm_finetuning_data)

2766

In [ ]:
# train_sample_sz = 2700
# train_ds = llm_finetuning_data[:train_sample_sz]
# eval_ds = llm_finetuning_data[train_sample_sz:]

# os.makedirs(join(data_dir, "datasets", "llamafactory-finetune-data"), exist_ok=True)

# with open(join(data_dir, "datasets", "llamafactory-finetune-data", "train.json"), "w") as dest:
#     json.dump(train_ds, dest, ensure_ascii=False, default=str)

# with open(join(data_dir, "datasets", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
#     json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [ ]:
join(data_dir,"llamafactory-finetune-data", "val.json")

'/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/val.json'

# Finetuning

In [ ]:
# # Configure LLaMA-Factory for the new datasets

# # update /content/LLaMA-Factory/data/dataset_info.json and append
# ```
"news_finetune_train": {
    "file_name": '/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/val.json',
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "system": "system",
        "history": "history"
    }
},
"news_finetune_val": {
    "file_name": "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/train.json",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "system": "system",
        "history": "history"
    }
}
# ```
   "news_finetune_train": {
        "file_name": "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }

# https://wandb.ai/mr-bakrianoo/llamafactory/runs/apwbkni9
# https://wandb.ai/mr-bakrianoo/llamafactory/runs/c5tf0q90

In [ ]:
import json
import os

dataset_info_path = '/content/LLaMA-Factory/data/dataset_info.json'

# Ensure the directory exists
os.makedirs(os.path.dirname(dataset_info_path), exist_ok=True)

# Try to load existing dataset_info.json, or start with an empty dictionary
existing_dataset_info = {}
if os.path.exists(dataset_info_path):
    try:
        with open(dataset_info_path, 'r', encoding='utf8') as f:
            existing_dataset_info = json.load(f)
    except json.JSONDecodeError:
        print(f"Warning: {dataset_info_path} is not a valid JSON. Overwriting it.")

# Define the custom dataset entries
custom_datasets = {
    "news_finetune_train": {
        "file_name": "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
}

# Update the existing dataset info with the custom datasets
existing_dataset_info.update(custom_datasets)

# Write the combined (or new) dataset info back to the file
with open(dataset_info_path, 'w', encoding='utf8') as f:
    json.dump(existing_dataset_info, f, ensure_ascii=False, indent=4)

print(f"Successfully updated '{dataset_info_path}' with news_finetune_train and news_finetune_val.")

Successfully updated '/content/LLaMA-Factory/data/dataset_info.json' with news_finetune_train and news_finetune_val.


In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /gdrive/MyDrive/youtube-resources/llm-finetuning/models/checkpoint-1500
output_dir: /gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/checkpoints
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: newsx-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "sherif-911/news-llm-analyzer"
# hub_private_repo: true
hub_strategy: checkpoint

Overwriting /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-06-25 23:34:46] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:771] 2026-06-25 23:34:46,826 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:847] 2026-06-25 23:34:46,831 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attent

# New Fintuning Model Evaluation

In [11]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

In [13]:
finetuned_model_id = "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/checkpoints"
model.load_adapter(finetuned_model_id)

In [16]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(translated_messages)#details_extraction_messages) #translated_messages

In [17]:
import json_repair

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

parse_json(response)

{'translated_title': 'The Role of Family in Financial Relationships',
 'translated_text': 'Forbes magazine reported that the family plays a pivotal role in shaping individuals\' relationship with money, as this relationship is influenced by inherited financial behaviors across generations.\n\nThe report, based on research by Professor Shane Everette on financial well-being, explains that each person has a \'financial personality\' determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\n\nThe three dimensions of the money relationship\nAccording to the study, there are three main dimensions that form our relationship with money:\n\nAcquisition (A): Individuals belonging to this dimension tend to view money as a commodity that can be accumulated, seeing wealth accumulation as a goal in itself. The downside of this pattern is the potential for it to turn into an obsession with wealth or vice versa, meaning complete rejection 

*italicized text*#### Tip for Qwen2.5

Qwen2.5 oftenly produce chinese characters with some responses. To skip this, use the next class to generate responses.

Source:
`https://jupyter267.medium.com/how-to-eliminate-the-chance-of-generating-chinese-in-qwen-2-5-2cf919bb0fdc`



In [22]:
class Generator:
    def __init__(self, model, tokenizer):

        self.model, self.tokenizer = model, tokenizer
        self.mask = None

    def generate(self, messages:list, max_new_tokens: int=2000, temperature:float=0.1):

        def logits_processor(token_ids, logits):
          # logits_processor default recieve the logits which is the score matrix of each time-step
          """
              A processor to ban Chinese character
          """
          if self.mask is None:
              # as we don't know where the Chinses tokens locate at which index
              # in the vocabulary but we know how it looks like and the range of it

              # decode all the tokens in the vocabulary in order
              token_ids = torch.arange(logits.size(-1))
              decoded_tokens = self.tokenizer.batch_decode(token_ids.unsqueeze(1), skip_special_tokens=True)

              # create a mask tensor to exclude positions of Chinese characters.
              # since this process uses a for loop and is time-consuming,
              # the result will be stored as a property for later use to ensure it only runs once.
              self.mask = torch.tensor([
                  # loop through each token in the vocabulary and compare it to Chinese characters.
                  any(0x4E00 <= ord(c) <= 0x9FFF or 0x3400 <= ord(c) <= 0x4DBF or 0xF900 <= ord(c) <= 0xFAFF for c in
                      token)
                  for token in decoded_tokens
              ])

          # mask the score by - inf
          logits[:, self.mask] = -float("inf")
          return logits

        # this step transforms the messages into a string,
        # adding special tokens e.g separate tokens between system content user queries
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            # add the logits_processor here
            logits_processor=[logits_processor]
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return response

# Cost Estimation

In [19]:
!pip install faker
from tqdm.auto import tqdm
from faker import Faker
import random
from datetime import datetime

start_time = datetime.now()
fake = Faker('ar')

input_tokens = 0
output_tokens = 0

for i in tqdm(range(30)):
    prompt = fake.text(max_nb_chars=random.randint(150, 200))

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    response = generate_resp(messages)

    input_tokens += len(tokenizer.apply_chat_template(messages))
    output_tokens += len(tokenizer.encode(response))

total_time = (datetime.now() - start_time).total_seconds()

print(f"Total Time: {total_time} seconds")
print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {input_tokens + output_tokens}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.1 MB/s eta 0:00:00


  0%|          | 0/30 [00:00<?, ?it/s]

Total Time: 580.829894 seconds
Input Tokens: 2469
Output Tokens: 10135
Total Tokens: 12604


In [20]:
12604 / 580

21.73103448275862

# VLLM

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_model_id = "/gdrive/MyDrive/youtube-resources/llm-finetuning/models"


!nohup vllm serve "{base_model_id}" --dtype=half --gpu-memory-utilization 0.8 --max_lora_rank 64 --enable-lora --lora-modules news-lora="{adapter_model_id}" &


In [ ]:
!tail -n 30 nohup.out

### Inference

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

prompt = tokenizer.apply_chat_template(
    translation_messages,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
vllm_model_id = "news-lora"

llm_response = requests.post("http://localhost:8000/v1/completions", json={
    "model": vllm_model_id,
    "prompt": prompt,
    "max_tokens": 1000,
    "temperature": 0.3
})

llm_response.json()

## Load Testing

In [ ]:
%%writefile locust.py

import random
import json
from locust import HttpUser, task, between, constant
from transformers import AutoTokenizer
from faker import Faker

fake = Faker('ar')

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        model_id = "news-lora"
        prompt = fake.text(max_nb_chars=random.randint(150, 200))

        message = {
            "model": model_id,
            "prompt": prompt,
            "max_tokens": 512,
            "temperature": 0.3
        }

        llm_response = self.client.post("/v1/completions", json=message)

        if llm_response.status_code == 200:
            with open("./vllm_tokens.txt", "a") as dest:
                dest.write(json.dumps({
                    "prompt": prompt,
                    "response": llm_response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")


In [ ]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

In [ ]:
vllm_tokens = [
    json.loads(line.strip())
    for line in open("./vllm_tokens.txt") if line.strip() != ""
]

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

total_input_tokens = sum([ len(tokenizer.encode(rec['prompt'])) for rec in vllm_tokens ])
total_output_tokens = sum([ len(tokenizer.encode(rec['response'])) for rec in vllm_tokens ])

print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")

In [21]:
37840 / 60

630.6666666666666

# README.md: News LLM Analyzer Notebook

This notebook demonstrates a comprehensive workflow for fine-tuning a Large Language Model (LLM) for news analysis tasks, including details extraction and translation. It covers data preparation, LLM-based data generation (knowledge distillation), LoRA fine-tuning using LLaMA-Factory, model evaluation, VLLM deployment for optimized inference, and load testing with Locust.

## Table of Contents
1. [Setup and Dependencies](#1-setup-and-dependencies)
2. [Data Preparation and Pydantic Models](#2-data-preparation-and-pydantic-models)
3. [Base Model Loading and Initial Evaluation](#3-base-model-loading-and-initial-evaluation)
4. [Knowledge Distillation (Data Generation for Fine-tuning)](#4-knowledge-distillation-data-generation-for-fine-tuning)
5. [Finetuning Dataset Formatting](#5-finetuning-dataset-formatting)
6. [LLaMA-Factory Integration and Fine-tuning](#6-llama-factory-integration-and-fine-tuning)
7. [New Finetuned Model Evaluation](#7-new-finetuned-model-evaluation)
8. [VLLM Deployment](#8-vllm-deployment)
9. [Inference with VLLM](#9-inference-with-vllm)
10. [Load Testing with Locust](#10-load-testing-with-locust)
11. [Qwen2.5 Chinese Character Tip](#11-qwen25-chinese-character-tip)

---

## 1. Setup and Dependencies
This section handles the initial environment setup, including mounting Google Drive and installing all necessary Python packages. Version pinning is used for critical libraries to ensure reproducibility.

- **Mount Google Drive**: Accesses files stored in your Google Drive.
- **Install Libraries**: Installs `transformers`, `peft`, `accelerate`, `datasets`, `openai`, `wandb`, `groq`, and `json_repair`. Note the specific version requirements and `llamafactory`'s installation.
- **LLaMA-Factory**: Clones the LLaMA-Factory repository and installs it in editable mode. This framework is used for efficient LLM fine-tuning.
- **Hugging Face and W&B Login**: Authenticates with Hugging Face Hub and Weights & Biases for model access and experiment tracking.

```python
# Mount Google Drive
from google.colab import drive
drive.mount('/gdrive')

# Install core dependencies (specific versions)
# !pip install -qU transformers==4.57.1 datasets==3.2.0 optimum==1.24.0
# !pip install -qU openai==1.61.0 wandb
# !pip install groq
# !pip install peft==0.17.1
# !pip install accelerate==1.8.1

# Install json_repair
!pip install json_repair

# Clone and install LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

# Login to Hugging Face and Weights & Biases
from google.colab import userdata
import wandb
from huggingface_hub import login

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('HF')
login(token=hf_token)
!huggingface-cli whoami
```

## 2. Data Preparation and Pydantic Models
This section defines the structure for extracting and translating news data using Pydantic models. It also sets up the prompting messages for the LLM.

- **Story Content**: A sample Arabic news story is provided as `story`.
- **Pydantic Models**:
    - `NewsDetails`: Defines the schema for extracting `story_title`, `story_keywords`, `story_summary`, `story_category`, and `story_entities` (with specific `EntityType`).
    - `TranslatedStory`: Defines the schema for `translated_title` and `translated_text` for translation tasks.
- **LLM Prompting Messages**: `details_extraction_messages` and `translated_messages` are created based on these Pydantic schemes to guide the LLM in generating structured JSON output.

```python
# Example of Pydantic models for structured data extraction and translation
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
import json

# ... (NewsDetails and TranslatedStory Pydantic model definitions as in notebook)

# Example of messages for LLM prompting
details_extraction_messages = [
    # ... (system and user messages for news details extraction)
]

translated_messages = [
    # ... (system and user messages for story translation)
]
```

## 3. Base Model Loading and Initial Evaluation
This part loads a base LLM (Qwen2.5-1.5B-Instruct) and demonstrates how to use it for inference directly.

- **Model Loading**: `AutoModelForCausalLM` and `AutoTokenizer` are used to load the pre-trained `Qwen/Qwen2.5-1.5B-Instruct` model.
- **`generate_resp` Function**: A helper function `generate_resp` is defined to abstract the model inference process, including applying chat templates and decoding responses.
- **`parse_json` Function**: A function using `json_repair` is provided to robustly parse potentially malformed JSON output from the LLM.
- **Initial Inference**: The base model is used to extract details and translate the `story`.

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda"
torch_dtype = None # or torch.bfloat16 if supported

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map= "auto",
    torch_dtype= torch_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def generate_resp(messages):
    # ... (implementation as in notebook)
    return response

import json_repair
def parse_json(text):
    # ... (implementation as in notebook)
    return None

# Example usage:
response_details = generate_resp(details_extraction_messages)
parsed_details = parse_json(response_details)

response_translation = generate_resp(translated_messages)
parsed_translation = parse_json(response_translation)
```

## 4. Knowledge Distillation (Data Generation for Fine-tuning)
This section outlines the process of generating a synthetic dataset for fine-tuning using a powerful cloud LLM (Groq's Llama-3.3-70b-versatile). This is a knowledge distillation step to create task-specific training data.

- **Raw Data Loading**: Loads a raw news dataset (`news-sample.jsonl`).
- **Groq API Calls**: Iterates through the raw data, uses the Groq API with the `llama-3.3-70b-versatile` model to perform details extraction and translation, based on the Pydantic schemas defined earlier.
- **Rate Limiting Handling**: Includes `time.sleep` to manage API rate limits.
- **Output Storage**: Saves the generated structured JSON responses into `stf.jsonl` files.
- **Cost Estimation**: Tracks prompt and completion tokens to estimate API costs.

```python
import time
from groq import Groq
from tqdm.auto import tqdm
from os.path import join
import random

# ... (cloud_model_id, price_per_1m_input_tokens, price_per_1m_output_tokens, groq_client initialization)

raw_data_path = join(data_dir, "news-sample.jsonl")
raw_data = []
# ... (load raw_data from jsonl file)

save_to_details = join(data_dir, "stf_details.jsonl")
save_to_translation = join(data_dir, "stf_translation.jsonl")

for story in tqdm(raw_data):
    # Generate details extraction messages and call Groq API
    # ... (similar to E9B9fotGCeHs cell in notebook)
    # Introduce delay: time.sleep(1.5)
    # Parse and save response to stf_details.jsonl

    # Generate translation messages and call Groq API
    # ... (similar to WU8iGDIbabTo cell in notebook)
    # Parse and save response to stf_translation.jsonl

# Print total cost
```

## 5. Finetuning Dataset Formatting
The generated `stf.jsonl` files are then formatted into a structure suitable for LLaMA-Factory's fine-tuning input.

- **System Message**: A generic system message is defined for the fine-tuned model.
- **Data Transformation**: Each record from the `stf.jsonl` is transformed into a dictionary with `system`, `instruction`, `input`, `output`, and `history` fields, aligning with LLaMA-Factory's expected format.
- **Splitting Data**: The formatted data is split into training and validation sets and saved as `train.json` and `val.json` in the specified directory.

```python
sft_data_path = join(data_dir, "stf.jsonl") # Assuming a combined sft.jsonl from previous step
llm_finetuning_data = []

system_message = """You are a professional NLP data parser. ...""" # as defined in notebook

for line in open(sft_data_path):
    # ... (load rec, format into llm_finetuning_data as in notebook)

random.Random(45).shuffle(llm_finetuning_data)

# Split and save to train.json and val.json
# ... (as in Tyufq0twOXCJ cell in notebook)
```

## 6. LLaMA-Factory Integration and Fine-tuning
This section integrates the prepared dataset with LLaMA-Factory for fine-tuning the Qwen2.5 base model using LoRA.

- **Update `dataset_info.json`**: Programmatically adds the paths and column mappings for the `news_finetune_train` and `news_finetune_val` datasets to LLaMA-Factory's configuration.
- **Create `news_finetune.yaml`**: Generates a YAML configuration file that specifies:
    - **Model**: `Qwen/Qwen2.5-1.5B-Instruct`
    - **Finetuning Method**: LoRA (`lora_rank: 64`, `lora_target: all`)
    - **Datasets**: `news_finetune_train` and `news_finetune_val`
    - **Training Parameters**: `per_device_train_batch_size`, `gradient_accumulation_steps`, `learning_rate`, `num_train_epochs`, etc.
    - **Output**: `output_dir` for checkpoints, `report_to: wandb`, and `push_to_hub` settings.
- **Run LLaMA-Factory Train**: Executes the fine-tuning process using `llamafactory-cli train` with the generated YAML configuration.

```python
# Update LLaMA-Factory's dataset_info.json
import json
import os

dataset_info_path = '/content/LLaMA-Factory/data/dataset_info.json'
# ... (code to update dataset_info.json as in df63434c cell)

# Create news_finetune.yaml configuration file
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml
# ... (YAML content as in 1B9O3TrGS11Z cell)

# Run fine-tuning
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml
```

## 7. New Finetuned Model Evaluation
After fine-tuning, the updated model (base model + LoRA adapter) is loaded and evaluated.

- **Load Base Model**: Reloads the `Qwen/Qwen2.5-1.5B-Instruct` base model.
- **Load Adapter**: Uses `model.load_adapter()` to load the fine-tuned LoRA weights from the `finetuned_model_id` (e.g., from `/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/checkpoints`).
- **Inference**: Uses the `generate_resp` function with the loaded adapter to perform inference on new inputs.

```python
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Load fine-tuned adapter
finetuned_model_id = "/gdrive/MyDrive/Fine-tuning LLM/Data/llamafactory-finetune-data/checkpoints"
model.load_adapter(finetuned_model_id)

# Example inference with fine-tuned model
response = generate_resp(translated_messages) # or details_extraction_messages
parsed_response = parse_json(response)
print(parsed_response)
```

## 8. VLLM Deployment
This section demonstrates how to deploy the fine-tuned model with VLLM for high-throughput and low-latency inference.

- **VLLM Serve Command**: Uses `nohup vllm serve` to run a VLLM server in the background, specifying the base model, data types, GPU memory utilization, and enabling LoRA with the fine-tuned adapter.
- **Log Monitoring**: `!tail -n 30 nohup.out` is used to check the VLLM server logs.

```python
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_model_id = "/gdrive/MyDrive/youtube-resources/llm-finetuning/models" # Or your actual checkpoints dir

!nohup vllm serve "{base_model_id}" --dtype=half --gpu-memory-utilization 0.8 --max_lora_rank 64 --enable-lora --lora-modules news-lora="{adapter_model_id}" & # Run in background

# Check logs after a short wait to ensure server starts
# !sleep 30 # wait for server to start
!tail -n 30 nohup.out
```

## 9. Inference with VLLM
Interacting with the locally deployed VLLM server for inference.

- **Tokenizer**: Loads the tokenizer for the base model.
- **Prompt Formatting**: Formats the input messages using `tokenizer.apply_chat_template`.
- **HTTP Request**: Sends a POST request to the VLLM server's `/v1/completions` endpoint with the formatted prompt and model ID.

```python
import requests
from transformers import AutoTokenizer # Assuming it's already imported above

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Example: using translated_messages for inference
prompt_vllm = tokenizer.apply_chat_template(
    translated_messages,
    tokenize=False,
    add_generation_prompt=True
)

vllm_model_id = "news-lora"
llm_response = requests.post("http://localhost:8000/v1/completions", json={
    "model": vllm_model_id,
    "prompt": prompt_vllm,
    "max_tokens": 1000,
    "temperature": 0.3
})

print(llm_response.json())
```

## 10. Load Testing with Locust
This section sets up and runs a load test on the VLLM server using Locust to evaluate its performance under stress.

- **Locust File (`locust.py`)**: A Python script is generated that defines a `HttpUser` class. This class simulates users sending POST requests to the VLLM completion endpoint with dynamically generated prompts (using `faker`).
- **Load Test Execution**: `!locust` command is used to run the load test in headless mode, specifying the host, number of users, spawn rate, test duration, and an HTML report output.
- **Token Calculation**: After the load test, `vllm_tokens.txt` (where responses are saved) is parsed to calculate total input and output tokens processed by VLLM.

```python
# Generate locust.py file
%%writefile locust.py
# ... (locust script content as in TYcuOUtB7q1Y cell)

# Run Locust load test
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

# Process VLLM tokens from load test
vllm_tokens = [
    json.loads(line.strip())
    for line in open("./vllm_tokens.txt") if line.strip() != ""
]

# Calculate and print total input/output tokens
total_input_tokens = sum([ len(tokenizer.encode(rec['prompt'])) for rec in vllm_tokens ])
total_output_tokens = sum([ len(tokenizer.encode(rec['response'])) for rec in vllm_tokens ])

print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")
```

## 11. Qwen2.5 Chinese Character Tip
Provides a custom `Generator` class with a `logits_processor` to prevent Qwen2.5 from generating Chinese characters, which can sometimes occur unexpectedly with multilingual models.

```python
# Generator class to ban Chinese characters
import torch
class Generator:
    def __init__(self, model, tokenizer):
        self.model, self.tokenizer = model, tokenizer
        self.mask = None

    def generate(self, messages:list, max_new_tokens: int=2000, temperature:float=0.1):
        def logits_processor(token_ids, logits):
            # ... (implementation as in ungxEdlh_chB cell)
            return logits
        
        # ... (rest of the generate method using logits_processor)
        return response

# Example usage:
# my_generator = Generator(model, tokenizer)
# response_no_chinese = my_generator.generate(messages)
```